In [2]:
import os
import sys
import copy
import pandas as pd
import numpy as np

# get the absolute path of the current file
current_dir = os.path.dirname(os.path.abspath("__file__"))
# get the project root (one level up from notebooks/)
project_root = os.path.abspath(os.path.join(current_dir, ".."))
# add project root to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)

In [30]:
%reload_ext autoreload
%autoreload 2

from utils.stock_utils import download_stock_data
from utils.kpi import CAGR, volatility, Sharpe, max_dd, Sortino, calamar
from utils.technical_indicators import ATR

In [55]:
mid_cap = ["PAYTM.NS", "NYKAA.NS", "VMM.NS", "MFSL.NS", "EXIDEIND.NS", "UPL.NS", "BSE.NS", "POLICYBZR.NS", "M&MFIN.NS", "MANKIND.NS", "NATIONALUM.NS", "ABFRL.NS", "ESCORTS.NS", "WAAREEENER.NS", "TIINDIA.NS", "NMDC.NS", "KALYANKJIL.NS", "SUZLON.NS", "UNIONBANK.NS", "SRF.NS"]

In [74]:
ohlcv_intraday = download_stock_data(mid_cap, 30, "5m")

Successfully downloaded 20/20 stocks.


In [75]:
# calculating ATR and rolling max price for each stock and consolidating this info by stock in a separate dataframe
ohlcv_dict = copy.deepcopy(ohlcv_intraday)
tickers_signal = {}
tickers_ret = {}
for ticker in mid_cap:
    print("calculating ATR and rolling max price for ",ticker)
    ohlcv_dict[ticker]["ATR"] = ATR(ohlcv_dict[ticker],20)
    ohlcv_dict[ticker]["roll_max_cp"] = ohlcv_dict[ticker]["High"].rolling(20).max()
    ohlcv_dict[ticker]["roll_min_cp"] = ohlcv_dict[ticker]["Low"].rolling(20).min()
    ohlcv_dict[ticker]["roll_max_vol"] = ohlcv_dict[ticker]["Volume"].rolling(20).max()
    ohlcv_dict[ticker].dropna(inplace=True)
    tickers_signal[ticker] = ""
    tickers_ret[ticker] = [0]

calculating ATR and rolling max price for  PAYTM.NS
calculating ATR and rolling max price for  NYKAA.NS
calculating ATR and rolling max price for  VMM.NS
calculating ATR and rolling max price for  MFSL.NS
calculating ATR and rolling max price for  EXIDEIND.NS
calculating ATR and rolling max price for  UPL.NS
calculating ATR and rolling max price for  BSE.NS
calculating ATR and rolling max price for  POLICYBZR.NS
calculating ATR and rolling max price for  M&MFIN.NS
calculating ATR and rolling max price for  MANKIND.NS
calculating ATR and rolling max price for  NATIONALUM.NS
calculating ATR and rolling max price for  ABFRL.NS
calculating ATR and rolling max price for  ESCORTS.NS
calculating ATR and rolling max price for  WAAREEENER.NS
calculating ATR and rolling max price for  TIINDIA.NS
calculating ATR and rolling max price for  NMDC.NS
calculating ATR and rolling max price for  KALYANKJIL.NS
calculating ATR and rolling max price for  SUZLON.NS
calculating ATR and rolling max price for 

In [76]:
# identifying signals and calculating daily return
for ticker in mid_cap:
    print("calculating returns for ", ticker)
    
    # FIX 1: Reset the signal and the list for EVERY ticker/run
    tickers_signal[ticker] = ""
    tickers_ret[ticker] = [0] # FIX 2: Start with [0] to account for the first row (index 0)
    
    df = ohlcv_dict[ticker] # Shorthand for readability

    for i in range(1, len(df)):
        if tickers_signal[ticker] == "":
            tickers_ret[ticker].append(0)
            # Breakout logic
            if df["High"].iloc[i] >= df["roll_max_cp"].iloc[i] and \
               df["Volume"].iloc[i] > 1.5 * df["roll_max_vol"].iloc[i-1]:
                tickers_signal[ticker] = "Buy"
            elif df["Low"].iloc[i] <= df["roll_min_cp"].iloc[i] and \
                 df["Volume"].iloc[i] > 1.5 * df["roll_max_vol"].iloc[i-1]:
                tickers_signal[ticker] = "Sell"
        
        elif tickers_signal[ticker] == "Buy":
            # Stop loss (ATR)
            if df["Low"].iloc[i] < df["Close"].iloc[i-1] - df["ATR"].iloc[i-1]:
                tickers_signal[ticker] = ""
                # Calculate return at stop loss price
                ret = ((df["Close"].iloc[i-1] - df["ATR"].iloc[i-1]) / df["Close"].iloc[i-1]) - 1
                tickers_ret[ticker].append(ret)
            # Reverse signal
            elif df["Low"].iloc[i] <= df["roll_min_cp"].iloc[i] and \
                 df["Volume"].iloc[i] > 1.5 * df["roll_max_vol"].iloc[i-1]:
                tickers_signal[ticker] = "Sell"
                tickers_ret[ticker].append((df["Close"].iloc[i] / df["Close"].iloc[i-1]) - 1)
            else:
                tickers_ret[ticker].append((df["Close"].iloc[i] / df["Close"].iloc[i-1]) - 1)
                
        elif tickers_signal[ticker] == "Sell":
            # Stop loss (ATR)
            if df["High"].iloc[i] > df["Close"].iloc[i-1] + df["ATR"].iloc[i-1]:
                tickers_signal[ticker] = ""
                # Calculate return for short position at stop loss
                ret = (df["Close"].iloc[i-1] / (df["Close"].iloc[i-1] + df["ATR"].iloc[i-1])) - 1
                tickers_ret[ticker].append(ret)
            # Reverse signal
            elif df["High"].iloc[i] >= df["roll_max_cp"].iloc[i] and \
                 df["Volume"].iloc[i] > 1.5 * df["roll_max_vol"].iloc[i-1]:
                tickers_signal[ticker] = "Buy"
                tickers_ret[ticker].append((df["Close"].iloc[i-1] / df["Close"].iloc[i]) - 1)
            else:
                tickers_ret[ticker].append((df["Close"].iloc[i-1] / df["Close"].iloc[i]) - 1)
                
    # Now lengths are guaranteed to match
    ohlcv_dict[ticker]["ret"] = np.array(tickers_ret[ticker])

calculating returns for  PAYTM.NS
calculating returns for  NYKAA.NS
calculating returns for  VMM.NS
calculating returns for  MFSL.NS
calculating returns for  EXIDEIND.NS
calculating returns for  UPL.NS
calculating returns for  BSE.NS
calculating returns for  POLICYBZR.NS
calculating returns for  M&MFIN.NS
calculating returns for  MANKIND.NS
calculating returns for  NATIONALUM.NS
calculating returns for  ABFRL.NS
calculating returns for  ESCORTS.NS
calculating returns for  WAAREEENER.NS
calculating returns for  TIINDIA.NS
calculating returns for  NMDC.NS
calculating returns for  KALYANKJIL.NS
calculating returns for  SUZLON.NS
calculating returns for  UNIONBANK.NS
calculating returns for  SRF.NS


In [77]:
# Assuming Indian Market 5m candles
# 75 candles per day (9:15 to 3:30) * 30 days = 2250
TF = 2250 

stats = {}

for ticker in mid_cap:
    df = ohlcv_dict[ticker]
    
    # We pass the 'ret' column we just calculated in your signal loop
    stats[ticker] = {
        "CAGR": CAGR(df, timeframe=TF, column='ret', is_price=False),
        "Sharpe": Sharpe(df, timeframe=TF, column='ret', is_price=False),
        "Max_DD": max_dd(df, column='ret', is_price=False),
        "Vol": volatility(df, timeframe=TF, column='ret', is_price=False)
    }

# Convert to DataFrame to see it clearly
stats_df = pd.DataFrame(stats).T
print(stats_df)

                   CAGR    Sharpe    Max_DD       Vol
PAYTM.NS       0.048134  0.465924  0.015531  0.038921
NYKAA.NS       0.009511 -0.514620  0.025374  0.039814
VMM.NS         0.048033  0.440437  0.027986  0.040943
MFSL.NS        0.007330 -1.218055  0.013867  0.018612
EXIDEIND.NS    0.071190  0.782365  0.013967  0.052648
UPL.NS         0.074914  0.798649  0.022506  0.056238
BSE.NS         0.069125  1.090034  0.010850  0.035894
POLICYBZR.NS   0.002973 -1.085330  0.018141  0.024902
M&MFIN.NS     -0.025643 -1.166369  0.044624  0.047706
MANKIND.NS     0.048558  0.618445  0.018008  0.030007
NATIONALUM.NS  0.082654  1.196557  0.022904  0.044004
ABFRL.NS       0.081130  1.038361  0.021050  0.049241
ESCORTS.NS     0.006856 -0.989908  0.017509  0.023380
WAAREEENER.NS  0.056191  0.466683  0.028687  0.056122
TIINDIA.NS    -0.018212 -1.750108  0.022083  0.027548
NMDC.NS        0.074577  0.839586  0.028084  0.053094
KALYANKJIL.NS  0.067276  1.084658  0.016932  0.034366
SUZLON.NS      0.017580 -0.4

In [78]:
# 1. Extract all 'ret' columns into one DataFrame
all_returns = pd.DataFrame()

for ticker in mid_cap:
    # Use the 'ret' column we calculated in the signal loop
    all_returns[ticker] = ohlcv_dict[ticker]["ret"]

# 2. Calculate the Portfolio Return
# We take the mean across all tickers for each 5-minute candle
all_returns['portfolio_ret'] = all_returns.mean(axis=1)

# 3. Create a clean DataFrame for the KPI functions
portfolio_df = all_returns[['portfolio_ret']].copy()

In [79]:
# Define your annualization factor (5-min candles in a year)
TF = 2250 

portfolio_stats = {
    "Portfolio CAGR": CAGR(portfolio_df, timeframe=TF, column='portfolio_ret', is_price=False),
    "Portfolio Sharpe": Sharpe(portfolio_df, timeframe=TF, column='portfolio_ret', is_price=False),
    "Portfolio Max DD": max_dd(portfolio_df, column='portfolio_ret', is_price=False),
    "Portfolio Vol": volatility(portfolio_df, timeframe=TF, column='portfolio_ret', is_price=False)
}

# Display results
for kpi, value in portfolio_stats.items():
    if "Sharpe" in kpi:
        print(f"{kpi}: {value:.2f}")
    else:
        print(f"{kpi}: {value:.2%}")

Portfolio CAGR: 3.64%
Portfolio Sharpe: 0.54
Portfolio Max DD: 0.38%
Portfolio Vol: 1.19%


In [81]:
# Baseline - BUY & HOLD for Nifty Midcap 100

import yfinance as yf
# 1. Download Data
ticker = "NIFTY_MIDCAP_100.NS"
data = yf.download(ticker, period="30d", interval="5m")

# 2. Clean Data (yfinance multi-index handling if necessary)
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

# 3. Calculate KPIs
# Since we are using 5min data, timeframe = 18900
# Since we downloaded ohlcv, is_price = True
TF=2250
cagr_val = CAGR(data, timeframe=TF, column='Close', is_price=True)
vol_val = volatility(data, timeframe=TF, column='Close', is_price=True)
sharpe_val = Sharpe(data, timeframe=TF, column='Close', is_price=True)
mdd_val = max_dd(data, column='Close', is_price=True)

# 4. Display Results
print(f"--- Statistics for {ticker} (Last 1 Month) ---")
print(f"CAGR:           {cagr_val:.2%}")
print(f"volatility:     {vol_val:.2%}")
print(f"Sharpe Ratio:   {sharpe_val:.2f}")
print(f"Max Drawdown:   {mdd_val:.2%}")

[*********************100%***********************]  1 of 1 completed

--- Statistics for NIFTY_MIDCAP_100.NS (Last 1 Month) ---
CAGR:           -5.44%
volatility:     7.10%
Sharpe Ratio:   -1.19
Max Drawdown:   10.15%
